In [20]:

import numpy as np
import pandas as pd
import joblib

from scipy import stats

from sklearn.model_selection import train_test_split,cross_val_score,GridSearchCV

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import OneHotEncoder,StandardScaler,FunctionTransformer

from sklearn.pipeline import Pipeline, make_pipeline

from sklearn.compose import ColumnTransformer

from sklearn.metrics.pairwise import rbf_kernel

from sklearn.tree import DecisionTreeRegressor

from sklearn.linear_model import LinearRegression

from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score


# Load Database

In [21]:

housing = pd.read_csv(
    "california_housing_prices.csv"
)

# Stratified Train / Test Split

In [22]:

housing["income_category"] = pd.cut(
    housing["median_income"],
    bins=[0., 1.5, 3., 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5]
)

train_set, test_set = train_test_split(
    housing,
    test_size=0.2,
    random_state=40,
    stratify=housing["income_category"]
)

train_set = train_set.drop(columns="income_category")
test_set = test_set.drop(columns="income_category")


# Separate Features & Target


In [23]:

train_features = train_set.drop(
    columns="median_house_value"
)

train_target = train_set["median_house_value"]

test_features = test_set.drop(
    columns="median_house_value"
)

test_target = test_set["median_house_value"]


# Feature Engineering Functions

In [24]:

def ratio_of_columns(X):
    return X[:, [0]] / X[:, [1]]


def ratio_feature_name(transformer, feature_names_in):
    return ["ratio"]



# Feature Engineering Pipelines

In [25]:

def ratio_pipeline():

    return make_pipeline(

        SimpleImputer(strategy="median"),

        FunctionTransformer(
            ratio_of_columns,
            feature_names_out=ratio_feature_name
        ),

        StandardScaler()

    )


heavy_tail_pipeline = make_pipeline(

    SimpleImputer(strategy="median"),

    FunctionTransformer(
        np.log,
        feature_names_out="one-to-one"
    ),

    StandardScaler()

)

housing_age_pipeline = FunctionTransformer(

    rbf_kernel,

    feature_names_out="one-to-one",

    kw_args={
        "Y": [[35]],
        "gamma": 0.1
    }

)

default_numeric_pipeline = make_pipeline(

    SimpleImputer(strategy="median"),

    StandardScaler()

)

categorical_pipeline = Pipeline([

    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),

    (
        "encoder",
        OneHotEncoder()
    )

])

In [26]:

all_transformations = ColumnTransformer(

    [

        (
            "bedrooms_per_room",
            ratio_pipeline(),
            ["total_bedrooms", "total_rooms"]
        ),

        (
            "rooms_per_house",
            ratio_pipeline(),
            ["total_rooms", "households"]
        ),

        (
            "people_per_house",
            ratio_pipeline(),
            ["population", "households"]
        ),

        (
            "log_features",
            heavy_tail_pipeline,
            [
                "total_bedrooms",
                "total_rooms",
                "population",
                "households",
                "median_income"
            ]
        ),

        (
            "housing_age_rbf",
            housing_age_pipeline,
            ["housing_median_age"]
        ),

        (
            "categorical",
            categorical_pipeline,
            ["ocean_proximity"]
        )

    ],

    remainder=default_numeric_pipeline

)


# Transform Training Data

In [ ]:

train_features_transformed = all_transformations.fit_transform(
    train_features
)


# Feature Names

In [ ]:
feature_names = all_transformations.get_feature_names_out()

# Linear Regression Pipeline

In [ ]:

linear_regression_pipeline = Pipeline(

    [

        (
            "preprocessing",
            all_transformations
        ),

        (
            "model",
            LinearRegression()
        )

    ]

)